In [1]:
!pip install -q faster-whisper gTTS
!ffmpeg -version | head -n 1
print("✅ Dependencies ready")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
✅ Dependencies ready


In [2]:
from google.colab import files
uploaded = files.upload()   # select transcription_pipeline.py AND service.py
print("Uploaded:", list(uploaded.keys()))

Saving service.py to service (2).py
Saving transcription_pipeline.py to transcription_pipeline (2).py
Uploaded: ['service (2).py', 'transcription_pipeline (2).py']


In [3]:
from gtts import gTTS

text = ("Hello and welcome. This is a test of the transcription pipeline. "
        "It converts audio into timestamped text for downstream use.")

# MP3
gTTS(text=text, lang="en").save("sample.mp3")

# Convert the same speech to WAV to test a second format
!ffmpeg -y -loglevel error -i sample.mp3 sample.wav

print("✅ Created sample.mp3 and sample.wav")

✅ Created sample.mp3 and sample.wav


In [4]:
!python transcription_pipeline.py sample.mp3 --word-timestamps -o out_mp3.json

2026-07-02 21:04:27,954 INFO Loading model 'base' (device=auto, compute=int8)
2026-07-02 21:04:28,101 INFO HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-base/revision/main "HTTP/1.1 200 OK"
2026-07-02 21:04:28,479 INFO Input: sample.mp3 | codec=mp3 sr=24000 ch=1 dur=9.2s
2026-07-02 21:04:28,639 INFO Processing audio with duration 00:09.216
2026-07-02 21:04:28,685 INFO VAD filter removed 00:00.000 of audio
2026-07-02 21:04:29,870 INFO Detected language 'en' with probability 0.99
2026-07-02 21:04:32,896 INFO Wrote out_mp3.json


In [5]:
import json
data = json.load(open("out_mp3.json"))
print("LANGUAGE:", data["language"])
print("FULL TEXT:", data["text"])
print("\nSEGMENTS (with timestamps):")
for s in data["segments"]:
    print(f"  [{s['start']:.2f}s → {s['end']:.2f}s] {s['text']}")
print("\nFirst 3 WORD timestamps:")
for w in data["segments"][0]["words"][:3]:
    print(f"  [{w['start']:.2f}s → {w['end']:.2f}s] {w['word']}")

LANGUAGE: en
FULL TEXT: Hello and welcome. This is a test of the transcription pipeline. It converts audio into time stamp text for downstream use.

SEGMENTS (with timestamps):
  [0.00s → 6.84s] Hello and welcome. This is a test of the transcription pipeline. It converts audio into time stamp
  [6.84s → 8.52s] text for downstream use.

First 3 WORD timestamps:
  [0.00s → 0.36s]  Hello
  [0.36s → 0.62s]  and
  [0.62s → 1.04s]  welcome.


In [6]:
!python transcription_pipeline.py sample.wav -o out_wav.json
import json
print("WAV transcript:", json.load(open("out_wav.json"))["text"])
print("✅ Second format (WAV) works → format handling verified")

2026-07-02 21:04:40,747 INFO Loading model 'base' (device=auto, compute=int8)
2026-07-02 21:04:40,854 INFO HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-base/revision/main "HTTP/1.1 200 OK"
2026-07-02 21:04:41,248 INFO Input: sample.wav | codec=pcm_s16le sr=24000 ch=1 dur=9.2s
2026-07-02 21:04:41,432 INFO Processing audio with duration 00:09.216
2026-07-02 21:04:41,477 INFO VAD filter removed 00:00.000 of audio
2026-07-02 21:04:42,646 INFO Detected language 'en' with probability 0.99
2026-07-02 21:04:45,448 INFO Wrote out_wav.json
WAV transcript: Hello and welcome. This is a test of the transcription pipeline. It converts audio into time stamp text for downstream use.
✅ Second format (WAV) works → format handling verified


In [11]:
import os, subprocess, json

# make sure the sample audio exists
if not os.path.exists("sample.wav"):
    from gtts import gTTS
    gTTS("Hello and welcome. This is a test of the transcription pipeline. "
         "It converts audio into timestamped text.", lang="en").save("sample.mp3")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i","sample.mp3","sample.wav"], check=True)

# build a longer file
with open("list.txt","w") as f:
    for _ in range(12): f.write("file 'sample.wav'\n")
subprocess.run(["ffmpeg","-y","-loglevel","error","-f","concat","-safe","0","-i","list.txt","long.wav"], check=True)

dur = float(subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                            "-of","csv=p=0","long.wav"], capture_output=True, text=True).stdout.strip())
print(f"long.wav = {dur:.1f}s  (chunking triggers above 15s)")

# force chunking
subprocess.run(["python","transcription_pipeline.py","long.wav","--max-chunk-seconds","15","-o","out_long.json"], check=True)

segs = json.load(open("out_long.json"))["segments"]
print(f"\n✅ Long-file chunking produced {len(segs)} segments")
if segs:
    print("First segment starts at:", round(segs[0]["start"],1), "s")
    print("Last segment ends at:  ", round(segs[-1]["end"],1), "s → timestamps span the whole file")
else:
    print("⚠️ Still empty — paste the full output here and I'll dig in")

long.wav = 110.6s  (chunking triggers above 15s)

✅ Long-file chunking produced 32 segments
First segment starts at: 0.0 s
Last segment ends at:   110.0 s → timestamps span the whole file


In [12]:
!pip install -q httpx
from fastapi.testclient import TestClient
from service import app

client = TestClient(app)

# Synchronous endpoint
r = client.post("/transcribe", files={"file": open("sample.mp3", "rb")})
print("POST /transcribe →", r.status_code)
print("Transcript:", r.json()["text"])

# Async job endpoint
job = client.post("/jobs", files={"file": open("sample.wav", "rb")}).json()
print("\nPOST /jobs →", job)
result = client.get(f"/jobs/{job['job_id']}").json()
print("GET /jobs/{id} → status:", result["status"])
print("✅ API works: sync + async job endpoints verified")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


POST /transcribe → 200
Transcript: Hello and welcome. This is a test of the transcription pipeline. It converts audio into time stamp text for downstream use.

POST /jobs → {'job_id': '13d21d4a2af1485dbb74681547a466e0', 'status': 'queued'}
GET /jobs/{id} → status: done
✅ API works: sync + async job endpoints verified
